# Healthcare Utilization, Cost & Demand Analytics Platform

## Phase 1–3: Business Understanding, Data Understanding, and Data Preparation

### Project Overview

Healthcare organizations generate large volumes of patient, encounter, provider, and cost data. Understanding how healthcare services are utilized and what factors drive healthcare costs is critical for effective decision-making.

This project aims to build an end-to-end Healthcare Analytics Platform using synthetic healthcare data generated through Synthea.

The project will focus on:

- Patient demographics
- Healthcare utilization patterns
- Cost analysis
- Provider and organization utilization
- Demand forecasting
- Executive dashboarding

This notebook covers the first three phases of the project:
1. Business Understanding
2. Data Understanding
3. Data Preparation & Feature Engineering

# Phase 1: Business Understanding

## Business Problem

Healthcare organizations need visibility into patient demographics, service utilization patterns, healthcare costs, and future demand trends.

Without analytical insights, healthcare administrators may struggle to:

- Allocate resources efficiently
- Understand cost drivers
- Monitor service utilization
- Anticipate future healthcare demand

## Project Objective

Develop a Healthcare Utilization, Cost & Demand Analytics Platform that enables healthcare administrators to make data-driven operational and financial decisions.

## Key Business Questions

1. Who are the patients utilizing healthcare services?
2. How are healthcare services being utilized?
3. What are the most common medical conditions?
4. What factors drive healthcare costs?
5. How has healthcare demand changed over time?
6. What healthcare demand can be expected in the future?

# Phase 2: Data Understanding

## Dataset Source

The dataset was generated using Synthea, an open-source synthetic healthcare data generator.

The dataset contains information about:

- Patients
- Healthcare encounters
- Medical conditions
- Healthcare providers
- Healthcare organizations

The goal of this phase is to understand the structure, quality, and relationships within the data before performing any transformations.

In [2]:
import pandas as pd
import numpy as np

## Loading Datasets

The first step is to load all source datasets and review their structure.

In [3]:
patients = pd.read_csv("patients.csv")
encounters = pd.read_csv("encounters.csv")
conditions = pd.read_csv("conditions.csv")
providers = pd.read_csv("providers.csv")
organizations = pd.read_csv("organizations.csv")

## Dataset Overview

Before performing any transformations, it is important to understand the size of each dataset.

This provides an initial understanding of the healthcare ecosystem represented in the data.

In [4]:
datasets = {
    "Patients": patients,
    "Encounters": encounters,
    "Conditions": conditions,
    "Providers": providers,
    "Organizations": organizations
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Patients: (1163, 25)
Encounters: (61459, 15)
Conditions: (38094, 6)
Providers: (5056, 12)
Organizations: (1127, 11)


## Data Quality Assessment

Data quality checks help identify:

- Missing values
- Duplicate records
- Data consistency issues

These checks ensure that the analytical datasets are reliable and suitable for downstream analysis.

In [5]:
patients.isnull().sum()

Id                        0
BIRTHDATE                 0
DEATHDATE              1000
SSN                       0
DRIVERS                 215
PASSPORT                276
PREFIX                  245
FIRST                     0
LAST                      0
SUFFIX                 1147
MAIDEN                  832
MARITAL                 384
RACE                      0
ETHNICITY                 0
GENDER                    0
BIRTHPLACE                0
ADDRESS                   0
CITY                      0
STATE                     0
COUNTY                    0
ZIP                     545
LAT                       0
LON                       0
HEALTHCARE_EXPENSES       0
HEALTHCARE_COVERAGE       0
dtype: int64

In [6]:
encounters.isnull().sum()

Id                         0
START                      0
STOP                       0
PATIENT                    0
ORGANIZATION               0
PROVIDER                   0
PAYER                      0
ENCOUNTERCLASS             0
CODE                       0
DESCRIPTION                0
BASE_ENCOUNTER_COST        0
TOTAL_CLAIM_COST           0
PAYER_COVERAGE             0
REASONCODE             45502
REASONDESCRIPTION      45502
dtype: int64

In [7]:
conditions.isnull().sum()

START             0
STOP           8169
PATIENT           0
ENCOUNTER         0
CODE              0
DESCRIPTION       0
dtype: int64

### Observations

Several demographic attributes contain missing values, which is expected in healthcare datasets.

The most notable missing values include:

- Death dates for living patients
- Optional demographic attributes
- Reason codes for certain encounters

These missing values do not significantly impact the planned analysis.

In [8]:
print("Patient duplicates:", patients['Id'].duplicated().sum())
print("Encounter duplicates:", encounters['Id'].duplicated().sum())

Patient duplicates: 0
Encounter duplicates: 0


### Observations

No duplicate patient identifiers or encounter identifiers were identified.

This confirms that the primary entities in the dataset maintain record uniqueness.

# Phase 3: Data Preparation & Feature Engineering

The objective of this phase is to transform raw healthcare records into analysis-ready datasets.

Transformations include:

- Standardizing column names
- Creating derived features
- Improving data usability
- Preparing datasets for SQL analysis and dashboard development

## Patients Dataset Preparation

The patients dataset contains demographic and financial information.

The following transformations will be performed:

- Rename columns
- Convert date fields
- Calculate patient age
- Create age groups
- Create a senior citizen indicator

In [9]:
patients = patients.rename(columns={
    'Id': 'patient_id',
    'BIRTHDATE': 'birth_date',
    'DEATHDATE': 'death_date',
    'GENDER': 'gender',
    'RACE': 'race',
    'ETHNICITY': 'ethnicity',
    'CITY': 'city',
    'STATE': 'state',
    'COUNTY': 'county',
    'HEALTHCARE_EXPENSES': 'total_healthcare_expenses',
    'HEALTHCARE_COVERAGE': 'total_healthcare_coverage'
})

In [10]:
patients['birth_date'] = pd.to_datetime(patients['birth_date'])
patients['death_date'] = pd.to_datetime(patients['death_date'])

In [11]:
today = pd.Timestamp.today()

patients['age'] = np.where(
    patients['death_date'].notna(),
    (patients['death_date'] - patients['birth_date']).dt.days / 365.25,
    (today - patients['birth_date']).dt.days / 365.25
)

patients['age'] = patients['age'].astype(int)

In [12]:
patients['age_group'] = pd.cut(
    patients['age'],
    bins=[0, 17, 34, 49, 64, 120],
    labels=['0-17', '18-34', '35-49', '50-64', '65+'],
    include_lowest=True
)

In [ ]:
patients['senior_flag'] = np.where(patients['age'] >= 65,'Yes','No')

In [14]:
patients_clean = patients[[
    'patient_id',
    'birth_date',
    'death_date',
    'age',
    'age_group',
    'senior_flag',
    'gender',
    'race',
    'ethnicity',
    'city',
    'state',
    'county',
    'total_healthcare_expenses',
    'total_healthcare_coverage'
]]

In [15]:
patients_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1163 entries, 0 to 1162
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   patient_id                 1163 non-null   object        
 1   birth_date                 1163 non-null   datetime64[ns]
 2   death_date                 163 non-null    datetime64[ns]
 3   age                        1163 non-null   int64         
 4   age_group                  1163 non-null   category      
 5   senior_flag                1163 non-null   object        
 6   gender                     1163 non-null   object        
 7   race                       1163 non-null   object        
 8   ethnicity                  1163 non-null   object        
 9   city                       1163 non-null   object        
 10  state                      1163 non-null   object        
 11  county                     1163 non-null   object        
 12  total_

In [16]:
patients_clean.head()

,patient_id,birth_date,death_date,age,age_group,senior_flag,gender,race,ethnicity,city,state,county,total_healthcare_expenses,total_healthcare_coverage
0,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,2019-02-17,NaT,7,0-17,No,M,white,nonhispanic,Springfield,Massachusetts,Hampden County,9039.1645,7964.1255
1,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,NaT,20,18-34,No,F,white,nonhispanic,Bellingham,Massachusetts,Norfolk County,402723.4150,14064.1350
2,339144f8-50e1-633e-a013-f361391c4cff,1998-05-11,NaT,28,18-34,No,M,white,nonhispanic,Boston,Massachusetts,Suffolk County,571935.8725,787.5375
3,d488232e-bf14-4bed-08c0-a82f34b6a197,2003-01-28,NaT,23,18-34,No,F,white,nonhispanic,Hingham,Massachusetts,Plymouth County,582557.8030,104782.2070
4,217f95a3-4e10-bd5d-fb67-0cfb5e8ba075,1993-12-23,NaT,32,18-34,No,M,black,nonhispanic,Revere,Massachusetts,Suffolk County,475826.8550,18067.0950


In [17]:
print("Shape:", patients_clean.shape)
print("\nAge Summary:")
print(patients_clean['age'].describe())

print("\nAge Groups:")
print(patients_clean['age_group'].value_counts())

print("\nGender:")
print(patients_clean['gender'].value_counts())

Shape: (1163, 14)

Age Summary:
count    1163.000000
mean       45.432502
std        24.582074
min         0.000000
25%        24.500000
50%        46.000000
75%        64.000000
max       114.000000
Name: age, dtype: float64

Age Groups:
age_group
65+      282
18-34    256
50-64    249
35-49    191
0-17     185
Name: count, dtype: int64

Gender:
gender
F    616
M    547
Name: count, dtype: int64


In [18]:
patients_clean.to_csv("patients_clean.csv",index=False)

print("\npatients_clean.csv saved successfully.")


patients_clean.csv saved successfully.


### Results

The patients dataset was successfully standardized and enriched with demographic features that will support population-level healthcare analysis.

## Encounters Dataset Preparation

The encounters dataset serves as the primary fact table for the project.

The following features are created:

- Year
- Month
- Quarter
- Day of Week
- Out-of-Pocket Cost

These features will support utilization analysis, cost analysis, and forecasting.

In [19]:
encounters = encounters.rename(columns={
    'Id': 'encounter_id',
    'START': 'encounter_start',
    'STOP': 'encounter_end',
    'PATIENT': 'patient_id',
    'ORGANIZATION': 'organization_id',
    'PROVIDER': 'provider_id',
    'PAYER': 'payer_id',
    'ENCOUNTERCLASS': 'encounter_type',
    'DESCRIPTION': 'encounter_description',
    'BASE_ENCOUNTER_COST': 'base_cost',
    'TOTAL_CLAIM_COST': 'total_claim_cost',
    'PAYER_COVERAGE': 'payer_coverage',
    'REASONCODE': 'reason_code',
    'REASONDESCRIPTION': 'reason_description'
})

encounters['encounter_start'] = pd.to_datetime(encounters['encounter_start'])
encounters['encounter_end'] = pd.to_datetime(encounters['encounter_end'])

encounters['year'] = encounters['encounter_start'].dt.year

encounters['month'] = encounters['encounter_start'].dt.month

encounters['month_name'] = encounters['encounter_start'].dt.month_name()

encounters['quarter'] = encounters['encounter_start'].dt.quarter

encounters['day_of_week'] = encounters['encounter_start'].dt.day_name()

encounters['weekday_weekend'] = np.where(encounters['encounter_start'].dt.weekday >= 5,'Weekend','Weekday')

encounters['out_of_pocket_cost'] = (encounters['total_claim_cost']- encounters['payer_coverage'])

encounters_clean = encounters[[
    'encounter_id',
    'patient_id',
    'organization_id',
    'provider_id',
    'payer_id',
    'encounter_start',
    'encounter_end',
    'year',
    'month',
    'month_name',
    'quarter',
    'day_of_week',
    'weekday_weekend',
    'encounter_type',
    'encounter_description',
    'base_cost',
    'total_claim_cost',
    'payer_coverage',
    'out_of_pocket_cost',
    'reason_code',
    'reason_description'
]]

print("Shape:", encounters_clean.shape)

print("\nEncounter Types:")
print(encounters_clean['encounter_type'].value_counts())

print("\nMissing Values:")
print(encounters_clean.isnull().sum())

print("\nCost Summary:")
print(encounters_clean['total_claim_cost'].describe())

encounters_clean.to_csv("encounters_clean.csv",index=False)

print("\nencounters_clean.csv saved successfully.")

Shape: (61459, 21)

Encounter Types:
encounter_type
wellness      24038
ambulatory    20124
outpatient    10837
urgentcare     2564
emergency      2168
inpatient      1728
Name: count, dtype: int64

Missing Values:
encounter_id                 0
patient_id                   0
organization_id              0
provider_id                  0
payer_id                     0
encounter_start              0
encounter_end                0
year                         0
month                        0
month_name                   0
quarter                      0
day_of_week                  0
weekday_weekend              0
encounter_type               0
encounter_description        0
base_cost                    0
total_claim_cost             0
payer_coverage               0
out_of_pocket_cost           0
reason_code              45502
reason_description       45502
dtype: int64

Cost Summary:
count     61459.000000
mean       4149.657952
std       10919.677889
min           0.000000
25%         12

### Encounter Duration Investigation

Encounter duration was initially evaluated as a potential analytical feature.

However, several synthetic records exhibited unrealistic durations spanning multiple years.

To avoid misleading results, encounter duration will not be used in subsequent analyses.

## Organizations Dataset Preparation

The organizations dataset contains information about healthcare facilities where patient encounters occur.

The dataset is cleaned and standardized to support organization-level utilization and revenue analysis.

Key transformations:

- Rename columns using business-friendly names.
- Retain analysis-relevant attributes.
- Create an analysis-ready organizations dataset.

In [20]:
organizations = organizations.rename(columns={
    'Id': 'organization_id',
    'NAME': 'organization_name',
    'CITY': 'city',
    'STATE': 'state',
    'REVENUE': 'revenue',
    'UTILIZATION': 'organization_utilization'
})

organizations_clean = organizations[[
    'organization_id',
    'organization_name',
    'city',
    'state',
    'revenue',
    'organization_utilization'
]]

print(organizations_clean.shape)
print(organizations_clean.isnull().sum())

organizations_clean.to_csv("organizations_clean.csv",index=False)

print("organizations_clean.csv saved successfully.")

(1127, 6)
organization_id             0
organization_name           0
city                        0
state                       0
revenue                     0
organization_utilization    0
dtype: int64
organizations_clean.csv saved successfully.


### Results

The organizations dataset was successfully standardized and prepared for organization utilization and revenue analysis.

## Providers Dataset Preparation

The providers dataset contains information about healthcare professionals and their specialties.

The dataset is cleaned and standardized to support provider utilization and workforce analysis.

Key transformations:

- Rename columns using business-friendly names.
- Retain analysis-relevant attributes.
- Create an analysis-ready providers dataset.

In [21]:
providers = providers.rename(columns={
    'Id': 'provider_id',
    'ORGANIZATION': 'organization_id',
    'NAME': 'provider_name',
    'GENDER': 'provider_gender',
    'SPECIALITY': 'specialty',
    'CITY': 'city',
    'STATE': 'state',
    'UTILIZATION': 'provider_utilization'
})

providers_clean = providers[[
    'provider_id',
    'organization_id',
    'provider_name',
    'provider_gender',
    'specialty',
    'city',
    'state',
    'provider_utilization'
]]

print(providers_clean.shape)
print(providers_clean.isnull().sum())

providers_clean.to_csv("providers_clean.csv",index=False)

print("providers_clean.csv saved successfully.")


(5056, 8)
provider_id             0
organization_id         0
provider_name           0
provider_gender         0
specialty               0
city                    0
state                   0
provider_utilization    0
dtype: int64
providers_clean.csv saved successfully.


### Results

The providers dataset was successfully standardized and prepared for provider utilization and specialty analysis.

## Conditions Dataset Preparation

The conditions dataset contains medical conditions associated with patient encounters.

The dataset is cleaned and standardized to support disease prevalence and population health analysis.

Key transformations:

- Rename columns using business-friendly names.
- Convert date fields to datetime format.
- Create a condition status indicator.

In [22]:
conditions = conditions.rename(columns={
    'START': 'condition_start',
    'STOP': 'condition_end',
    'PATIENT': 'patient_id',
    'ENCOUNTER': 'encounter_id',
    'CODE': 'condition_code',
    'DESCRIPTION': 'condition_name'
})

conditions['condition_start'] = pd.to_datetime(conditions['condition_start'])

conditions['condition_end'] = pd.to_datetime(conditions['condition_end'])

conditions['condition_status'] = conditions['condition_end'].apply(lambda x: 'Active' if pd.isna(x) else 'Resolved')

conditions_clean = conditions[[
    'patient_id',
    'encounter_id',
    'condition_code',
    'condition_name',
    'condition_start',
    'condition_end',
    'condition_status'
]]

print(conditions_clean.shape)
print(conditions_clean.isnull().sum())

print("\nCondition Status:")
print(conditions_clean['condition_status'].value_counts())

conditions_clean.to_csv("conditions_clean.csv",index=False)

print("conditions_clean.csv saved successfully.")

(38094, 7)
patient_id             0
encounter_id           0
condition_code         0
condition_name         0
condition_start        0
condition_end       8169
condition_status       0
dtype: int64

Condition Status:
condition_status
Resolved    29925
Active       8169
Name: count, dtype: int64
conditions_clean.csv saved successfully.


### Results

The conditions dataset was successfully standardized and enriched with a condition status feature for future condition and disease analysis.

# Phase 3 Summary

The data preparation phase transformed raw healthcare records into analysis-ready datasets.

Final datasets created:

- patients_clean.csv
- encounters_clean.csv
- organizations_clean.csv
- providers_clean.csv
- conditions_clean.csv

These datasets will be used in subsequent phases for SQL analysis, exploratory data analysis, forecasting, and dashboard development.